<h1>Word Embedding Project</h1>

Imports Required library

In [152]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken

Load Text File

In [153]:
def load_small_sample(file_path, max_lines=2000):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = []
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            lines.append(line.strip())
    return "\n".join(lines)

raw_text = load_small_sample("hi_part_1.txt")

print("First 300 characters:\n")
print(raw_text[:300])

First 300 characters:

शारदा पारा के मिलन चैiक से आज महापौर देवेन्द्र यादव, पार्षद छोटे लाल चैधरी के साथ वार्ड का भ्रमण करते हुए वार्ड की पेयजल, स्वच्छता सफाई, विकास कार्यों का निरीक्षण किया।
सूचना का अधिकार - विभाग द्वारा तैयार 17 column सम्बंधित पंजी ,कार्यालय नगर पालिक निगम भिलाई जोन-06 रिसाली
आज सम्पत्तिकर अधिकारी एच.


Tokenization (Convert Text → Tokens)

In [154]:
tokenizer = tiktoken.get_encoding("gpt2")

tokens = torch.tensor(tokenizer.encode(raw_text), dtype=torch.long)

print("Total Tokens:", len(tokens))
print("First 20 Tokens:", tokens[:20])

Total Tokens: 866225
First 20 Tokens: tensor([11976,   114, 48077, 11976,   108, 11976,    99, 48077, 28225,   103,
        48077, 11976,   108, 48077, 28225,   243, 24231,   229, 28225,   106])


GPT Dataset (Efficient Version)

In [155]:
class GPTDataset(Dataset):
    def __init__(self, tokens, max_length=64, stride=32):
        self.tokens = tokens
        self.max_length = max_length
        self.stride = stride
        self.num_samples = (len(tokens) - max_length) // stride

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start = idx * self.stride
        end = start + self.max_length
        
        x = self.tokens[start:end]
        y = self.tokens[start+1:end+1]
        
        return x, y

Create Dataset & DataLoader

In [156]:
max_length = 64
stride = 32
batch_size = 8

dataset = GPTDataset(tokens, max_length=max_length, stride=stride)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True
)

print("Total Samples:", len(dataset))

Total Samples: 27067


Device Setup (CPU / GPU)

In [157]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


Create Embedding Layer

In [158]:
vocab_size = tokenizer.n_vocab
embedding_dim = 128

embedding = torch.nn.Embedding(vocab_size, embedding_dim).to(device)

print("Embedding Matrix Shape:", embedding.weight.shape)

Embedding Matrix Shape: torch.Size([50257, 128])


Create Output Linear Layer (LM Head)

In [159]:
lm_head = torch.nn.Linear(embedding_dim, vocab_size).to(device)

print("LM Head Created Successfully")

LM Head Created Successfully


Forward Pass (Test Run)

In [160]:
for x, y in dataloader:
    x = x.to(device)
    y = y.to(device)

    # Step 1: Convert tokens to embeddings
    x_emb = embedding(x)

    # Step 2: Convert embeddings to vocabulary logits
    logits = lm_head(x_emb)

    print("Input shape:", x.shape)
    print("Embedding shape:", x_emb.shape)
    print("Logits shape:", logits.shape)
    
    break

Input shape: torch.Size([8, 64])
Embedding shape: torch.Size([8, 64, 128])
Logits shape: torch.Size([8, 64, 50257])


Decode One Batch (Input vs Target Text)

In [161]:
batch = next(iter(dataloader))
inputs, targets = batch

print("INPUT TEXT:\n")
print(tokenizer.decode(inputs[0].tolist()))

print("\n" + "="*80 + "\n")

print("TARGET TEXT:\n")
print(tokenizer.decode(targets[0].tolist()))

INPUT TEXT:

 भी मज़ेदार वाकया होता तो उनके उद्दगार होत�


TARGET TEXT:

�ी मज़ेदार वाकया होता तो उनके उद्दगार होते


Token-by-Token Mapping (Text Level)

In [162]:
input_ids = inputs[0]
target_ids = targets[0]

print("First 5 Token Predictions:\n")

for i in range(5):
    print(
        tokenizer.decode([input_ids[i].item()]),
        "→",
        tokenizer.decode([target_ids[i].item()])
    )

First 5 Token Predictions:

 � → �
� → �
� → �
� →  �
 � → �


Token-by-Token Mapping (ID Level)

In [163]:
print("Token ID Mapping:\n")

for i in range(5):
    print(input_ids[i].item(), "→", target_ids[i].item())

Token ID Mapping:

28225 → 255
255 → 24231
24231 → 222
222 → 28225
28225 → 106


Progressive Sequence Prediction

In [164]:
print("Progressive Context Prediction:\n")

for i in range(10, 16):
    print(
        tokenizer.decode(input_ids[:i].tolist()),
        "\n→",
        tokenizer.decode(target_ids[:i].tolist()),
        "\n" + "-"*60
    )

Progressive Context Prediction:

 भी मज़ 
→ �ी मज़� 
------------------------------------------------------------
 भी मज़� 
→ �ी मज़े 
------------------------------------------------------------
 भी मज़े 
→ �ी मज़े� 
------------------------------------------------------------
 भी मज़े� 
→ �ी मज़ेद 
------------------------------------------------------------
 भी मज़ेद 
→ �ी मज़ेदा 
------------------------------------------------------------
 भी मज़ेदा 
→ �ी मज़ेदा� 
------------------------------------------------------------


Show Tensor Shapes Clearly

In [165]:
print("INPUT SHAPE:", inputs.shape)
print("TARGET SHAPE:", targets.shape)

print("\nSingle Sample Shape:", input_ids.shape)

INPUT SHAPE: torch.Size([8, 64])
TARGET SHAPE: torch.Size([8, 64])

Single Sample Shape: torch.Size([64])
